# Stage 1 - Stage offline assets (connected)

**NASA Space Apps 2026 - MODIS/VIIRS hotspot harmonization**

Resolve compatible dependencies, stage Linux wheels and any permitted model checkpoints/data, and generate hashes + a staging manifest. This is the ONLY stage with network access; everything later is airgapped.

| | |
|---|---|
| Stage | 1 of 6 - connected staging |
| Network required | YES - package index + approved data/model downloads only |
| Runs | the committed CLI scripts (`scripts/`) - logic stays out of cells |
| Writes to | `wheelhouse/ (Kaggle dataset or versioned volume, NOT Git)` |

Rules that make this notebook reproducible (docs/DEVELOPMENT.md): record the
environment below, keep every output under the working directory, never fetch
from PyPI/GitHub/Hugging Face/REST APIs during offline compute, and keep
generated weights/datasets out of Git.


## 0. Record the environment

Runtime discovery is authoritative over planning notes (docs/DEVELOPMENT.md).

In [ ]:
import sys, platform, subprocess
from pathlib import Path

# Kaggle working dir is the ONLY writable location during offline compute.
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "out"
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()

print("python   :", sys.version)
print("platform :", platform.platform())
print("machine  :", platform.machine())
print("repo     :", REPO_ROOT)
print("work dir :", WORK_DIR)

RECORD = {"python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine()}


## 1. Resolve and download compatible wheels

Pin exact versions only after this command has passed in a clean environment; `requirements-offline.txt` is generated from the package metadata.

In [ ]:
from pathlib import Path
WHEELHOUSE = WORK_DIR / "wheelhouse"
WHEELHOUSE.mkdir(parents=True, exist_ok=True)

# Produce a pinned requirements file from pyproject dependencies.
req = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True
).stdout

# Keep only the project's declared dependencies (numpy, pandas, lightgbm,
# PyYAML, joblib, torch, pytest). Adjust the filter if dependencies change.
KEEP = ("numpy", "pandas", "lightgbm", "PyYAML", "yaml", "joblib", "torch", "pytest")
lines = [l for l in req.splitlines() if l.split("==")[0].lower() in map(str.lower, KEEP)]
(Path(WORK_DIR) / "requirements-offline.txt").write_text("\n".join(lines))
print("\n".join(lines))

# Download Linux wheels for the target platform (no sdists: offline safety).
proc = subprocess.run(
    [sys.executable, "-m", "pip", "download", "-r", str(WORK_DIR / "requirements-offline.txt"),
     "--only-binary=:all:", "--dest", str(WHEELHOUSE)],
    capture_output=True, text=True)
print(proc.stdout[-3000:] or proc.stderr[-3000:])
assert proc.returncode == 0, "wheel download failed; resolve compatibility before proceeding"


## 2. Hash the wheelhouse and any staged data/checkpoints

Every staged file gets a SHA-256 recorded in the staging manifest; the quarantine notebook re-verifies these hashes offline.

In [ ]:
import hashlib, json, datetime
from pathlib import Path

def hash_tree(base: Path):
    return {
        str(p.relative_to(base)): sha256(p)
        for p in sorted(base.rglob("*"))
        if p.is_file()
    }

staging_manifest = {
    "schema_version": "1.0.0",
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "environment": RECORD,
    "wheelhouse": hash_tree(WHEELHOUSE),
    "notes": [
        "Verify data/model terms before staging; record source + license per file.",
        "Top-level package downloads do NOT guarantee all transitive dependencies; "
        "the quarantine stage proves installability offline.",
    ],
}
(WORK_DIR / "staging_manifest.json").write_text(json.dumps(staging_manifest, indent=2, sort_keys=True))
print(f"{len(staging_manifest['wheelhouse'])} files hashed into staging_manifest.json")


## 3. Export the staged assets

Package `wheelhouse/`, `requirements-offline.txt`, and `staging_manifest.json` as a Kaggle dataset (or another versioned volume). **Never commit wheels or data to Git** (README: compute and artifact separation).